# Notebook 2 — Neural networks in PyTorch

After [Notebook 1 — Setup](01_setup.ipynb), this notebook teaches you how to define, instantiate, and use a small network with `torch.nn`.

Run the code cells to try the APIs yourself.

Use the `pong-pytorch` kernel (project `.venv`).

## Running this notebook

**Locally**: the environment comes from the repo. `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

**On Colab**: the kernel exists before the project does, so the cell below clones the
repo and installs what Colab is missing. Run it first.

It installs `gymnasium` and `ale-py` only. Colab already provides a working CUDA build
of torch, and replacing it with the exact locked one means downloading the entire CUDA
stack for no benefit to anything taught here. The README describes the stricter route
if you ever need bit-identical versions.


In [ ]:
# Colab bootstrap. Does nothing when you run this notebook locally.
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The -d test makes re-running this cell harmless.
    ![ -d /content/pong-pytorch ] || git clone -q https://github.com/piyushahuja/pong-pytorch.git /content/pong-pytorch
    %cd /content/pong-pytorch
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    !~/.local/bin/uv pip install -q --system --python {sys.executable} gymnasium ale-py
    !~/.local/bin/uv pip install -q --system --python {sys.executable} -e . --no-deps

print("python:", sys.version.split()[0])
if IN_COLAB:
    print("working directory:", os.getcwd())


In [ ]:
# Confirm the environment is what you think it is.
import torch
import gymnasium
import ale_py

print("torch:     ", torch.__version__)
print("gymnasium: ", gymnasium.__version__)
print("ale-py:    ", ale_py.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [17]:
import math
from pathlib import Path

import torch
import torch.nn as nn

print(torch.__version__)
import pong        # PROJECT_ROOT: paths that work from anywhere


2.14.0


## What a two-layer net looks like

![Two-layer network: inputs to hidden units to one output](../assets/nn-two-layer.png)

Read the picture as code:

| In the figure | In PyTorch |
|---|---|
| Left (many inputs) | `input_dim` — length of the input vector |
| Blue circles | `hidden_dim` — `nn.Linear(input_dim, hidden_dim)` | 
| Pink circle | `output_dim` — here `1`, often via `nn.Linear(hidden_dim, 1)` |
| Lines | learnable weights inside those `Linear` layers |

If the single output is passed through a sigmoid, it becomes a number in $(0,1)$.

Blue circles are neurons. Each neuron is connected to each coordinate of the input vector by a weight. The number of weights are thus input_dim x hidden_dim. 
Another way to see this is the input vector and the neurons define a matrix of dimension hidden_dim x input_dim, where each row corresponds to a neuron and each column to a coordinate of a hidden_neuron. 

## Tensors and shapes

`nn.Linear(in_features, out_features)` is one linear layer of neurons that takes in input featues and outputs outpur features.


- `(features,)` — one example, no batch axis
- `(1, features)` — batch of size 1
- `(B, features)` — batch of size `B`

`unsqueeze` inserts a size-1 dimension. `squeeze` removes size-1 dimensions.

In [3]:
features = 8
x = torch.randn(features)
print("vector:", x.shape)

x_batched = x.unsqueeze(0)          # add a batch dim at front
print("unsqueeze(0):", x_batched.shape)

x_col = x.unsqueeze(-1)             # add a dim at the end
print("unsqueeze(-1):", x_col.shape)

print("squeeze back:", x_batched.squeeze(0).shape)
print("squeeze(-1):", x_col.squeeze(-1).shape)

vector: torch.Size([8])
unsqueeze(0): torch.Size([1, 8])
unsqueeze(-1): torch.Size([8, 1])
squeeze back: torch.Size([8])
squeeze(-1): torch.Size([8])


In [4]:
linear = nn.Linear(in_features=8, out_features=3)

ok = torch.randn(8)
print("ok input", ok.shape, "->", linear(ok).shape)

ok_batch = torch.randn(4, 8)
print("ok batch", ok_batch.shape, "->", linear(ok_batch).shape)

# Deliberate mistake: last dim is 7, but Linear expects 8
try:
    linear(torch.randn(7))
except RuntimeError as e:
    print("shape error (expected):")
    print(e)

ok input torch.Size([8]) -> torch.Size([3])
ok batch torch.Size([4, 8]) -> torch.Size([4, 3])
shape error (expected):
mat1 and mat2 shapes cannot be multiplied (1x7 and 8x3)


## Define, instantiate, call

1. Subclass `nn.Module`
2. Call `super().__init__()`
3. Register layers as attributes (`self.layer1 = ...`)
4. Implement `forward`
5. Instantiate with `model = Net(...)`
6. Call with `y = model(x)` (this runs `forward`)

In [5]:
class TinyNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        return x


model = TinyNet(input_dim=8, hidden_dim=5, output_dim=3)
print(model)

y = model(torch.randn(8))
print("output shape:", y.shape)
print("output:", y)

TinyNet(
  (layer1): Linear(in_features=8, out_features=5, bias=True)
  (layer2): Linear(in_features=5, out_features=3, bias=True)
)
output shape: torch.Size([3])
output: tensor([-0.1048,  0.4247, -0.3699], grad_fn=<ViewBackward0>)


## A concrete two-layer network

Same pattern as the figure, with optional dropout between layers.

- Several outputs → often treated as logits. Softmax turns them into a distribution over options.
- One output + sigmoid → one number in $(0,1)$.

In [24]:
class TwoLayerNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.0):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = self.dropout(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        return x


# Many outputs → logits
multi = TwoLayerNet(input_dim=16, hidden_dim=8, output_dim=4, dropout=0.1)
logits = multi(torch.randn(16))
probs = nn.functional.softmax(logits, dim=-1)
print("logits", logits.shape, logits)
print("softmax", probs.shape, probs, "sum=", probs.sum().item())

# One output → optional sigmoid
single = TwoLayerNet(input_dim=16, hidden_dim=8, output_dim=1)
logit = single(torch.randn(16))
p = torch.sigmoid(logit).squeeze(-1)
print("logit", logit.shape, "-> sigmoid squeezed", p.shape, p.item())

logits torch.Size([4]) tensor([ 0.3603, -0.1931, -0.4150, -0.1684], grad_fn=<ViewBackward0>)
softmax torch.Size([4]) tensor([0.3810, 0.2190, 0.1755, 0.2245], grad_fn=<SoftmaxBackward0>) sum= 1.0
logit torch.Size([1]) -> sigmoid squeezed torch.Size([]) 0.5549334287643433


## Interpretation examples (shapes only)

A network is just numbers in → numbers out. You *can* choose `input_dim` / `output_dim` so those numbers are easy to *read as* preferences over moves in a game.

The cells below do not play a game or learn anything. They only show shapes and one forward pass on fake inputs.

### Pong-shaped outputs (matches the figure)

Flattened screen features in; one number out. After sigmoid, you *could* interpret it as “prefer UP” (high) vs “prefer DOWN” (low).

In [7]:
class PongShapedNet(nn.Module):
    def __init__(self, input_dim=6400, hidden_dim=32):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = nn.functional.relu(self.layer1(x))
        x = torch.sigmoid(self.layer2(x))
        return x.squeeze(-1)


pong_net = PongShapedNet()
fake_frame = torch.randn(6400)          # stand-in for flattened pixels
prefer_up = pong_net(fake_frame)
print("input", fake_frame.shape)
print("output", prefer_up.shape, prefer_up.item(), "(interpret as prefer UP)")

input torch.Size([6400])
output torch.Size([]) 0.4823397099971771 (interpret as prefer UP)


### Chess-shaped outputs

Board features in; many outputs. Softmax → a distribution over move slots. Here we use a toy size (`64`) so the demo stays small; a fuller encoding of chess moves would use a much larger `output_dim`.

In [ ]:
BOARD_FEATURES = 64          # toy: one number per square
MOVE_SLOTS = 64              # toy: prefer which square / slot

chess_net = TwoLayerNet(
    input_dim=BOARD_FEATURES,
    hidden_dim=32,
    output_dim=MOVE_SLOTS,
)

fake_board = torch.randn(BOARD_FEATURES)
move_logits = chess_net(fake_board)
move_probs = nn.functional.softmax(move_logits, dim=-1)

print("input", fake_board.shape)
print("logits", move_logits.shape)
print("softmax", move_probs.shape, "sum=", move_probs)
print("most preferred slot:", int(move_probs.argmax()))

input torch.Size([64])
logits torch.Size([64])
softmax torch.Size([64]) sum= 1.0
most preferred slot: 51


### Go-shaped outputs

Board features in; one score per intersection (+ optional pass). Softmax → preference over board points. Toy board size `5×5` here; full Go is often `19×19 + 1`.

In [9]:
N = 5
GO_FEATURES = N * N
GO_MOVES = N * N + 1         # intersections + pass

go_net = TwoLayerNet(
    input_dim=GO_FEATURES,
    hidden_dim=32,
    output_dim=GO_MOVES,
)

fake_go = torch.randn(GO_FEATURES)
go_logits = go_net(fake_go)
go_probs = nn.functional.softmax(go_logits, dim=-1)

print("input", fake_go.shape)
print("logits", go_logits.shape, "(= board points + pass)")
print("softmax sum", go_probs.sum().item())
choice = int(go_probs.argmax())
if choice == GO_MOVES - 1:
    print("most preferred: pass")
else:
    print("most preferred intersection:", choice, "-> row/col", divmod(choice, N))

input torch.Size([25])
logits torch.Size([26]) (= board points + pass)
softmax sum 1.0
most preferred intersection: 7 -> row/col (1, 2)


## Building blocks

### Dropout

Randomly zeros a fraction of units during training so the net does not rely on single neurons. In `model.eval()`, dropout is turned off.

In [15]:
drop_net = TwoLayerNet(input_dim=8, hidden_dim=16, output_dim=4, dropout=0.5)
x = torch.randn(8)

drop_net.train()
y_train = torch.stack([drop_net(x) for _ in range(3)])
print("train mode (stochastic):\n", y_train)

drop_net.eval()
with torch.no_grad():
    y_eval = torch.stack([drop_net(x) for _ in range(3)])
print("eval mode (stable):\n", y_eval)

train mode (stochastic):
 tensor([[ 0.1487, -0.0278, -0.1912,  0.2247],
        [ 0.1360,  0.0066, -0.2259,  0.2192],
        [ 0.1360,  0.0066, -0.2259,  0.2192]], grad_fn=<StackBackward0>)
eval mode (stable):
 tensor([[ 0.1583,  0.0740, -0.3562,  0.2757],
        [ 0.1583,  0.0740, -0.3562,  0.2757],
        [ 0.1583,  0.0740, -0.3562,  0.2757]])


### ReLU vs LeakyReLU

ReLU is $\max(0, z)$. Units that always see negative inputs can get stuck at 0 (“dying ReLU”).

LeakyReLU allows a small slope for negatives so a weak gradient can still flow.

In [19]:
z = torch.tensor([-2.0, -0.5, 0.0, 1.5])
print("z         ", z.tolist())
print("relu      ", nn.functional.relu(z).tolist())
print("leaky_relu", nn.functional.leaky_relu(z, negative_slope=0.01).tolist())

z          [-2.0, -0.5, 0.0, 1.5]
relu       [0.0, 0.0, 0.0, 1.5]
leaky_relu [-0.019999999552965164, -0.004999999888241291, 0.0, 1.5]


### BatchNorm and weight init

BatchNorm (`nn.BatchNorm1d`) renormalizes along the batch for each feature — useful in larger nets.

Init sets starting weights. Common choices: Xavier uniform, or Normal with `std = 1 / sqrt(fan_in)`.

In [22]:
layer = nn.Linear(16, 8)
nn.init.xavier_uniform_(layer.weight)
nn.init.zeros_(layer.bias)
print("xavier weight std~", layer.weight.std().item())

layer2 = nn.Linear(16, 8, bias=False)
nn.init.normal_(layer2.weight, mean=0.0, std=1.0 / math.sqrt(16))
print("fan-in normal std~", layer2.weight.std().item())

# BatchNorm expects (batch, features)
bn = nn.BatchNorm1d(8)
batch = torch.randn(32, 8)
print("batchnorm", batch.shape, "->", bn(batch).shape)

xavier weight std~ 0.28657591342926025
fan-in normal std~ 0.24130596220493317
batchnorm torch.Size([32, 8]) -> torch.Size([32, 8])


## Device and checkpoints

Put both the model and its inputs on the same device. Save / load weights with `state_dict`.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

net = TwoLayerNet(8, 16, 4).to(device)
x = torch.randn(8, device=device)
print("forward on device:", net(x).shape)

ckpt = pong.PROJECT_ROOT / "assets" / "_demo_two_layer.pt"   # not Path("assets"): the notebook is not the root
torch.save(net.state_dict(), ckpt)

restored = TwoLayerNet(8, 16, 4).to(device)
restored.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
restored.eval()
print("restored forward:", restored(x).shape)
print("saved to", ckpt)

## What to try next

- Change `hidden_dim` (e.g. 8, 32, 128) and watch output shapes stay tied to `output_dim`.
- Pass a batch `(B, input_dim)` into your net and confirm the batch size is preserved.
- Practice `unsqueeze` / `squeeze` until shape errors feel readable.
- Scale inputs to a similar range (e.g. roughly $[0,1]$ or zero-mean) when you work with real data.
- Later, when you train: `loss.backward()` then `optimizer.step()` — not covered here on purpose.

You now have the core PyTorch pieces: Module → instantiate → forward → shapes.